# LangChain: Q&A over Documents

An example might be a tool that would allow you to query a product catalog for items of interest.

In [ ]:
#pip install --upgrade langchain

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Install LangChain packages compatible with the updated import paths used below
!pip install -qU langchain langchain-openai langchain-community langchain-classic docarray


**Important:** After the package installation cell finishes, restart the Colab session once if Colab still shows old import errors, then run the notebook from the top.

In [ ]:
import os
import warnings
from getpass import getpass

warnings.filterwarnings('ignore')

OPENAI_API_KEY = None

# Try Colab Secrets first. Fall back gracefully outside Colab or when the secret is missing.
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY1")
except Exception:
    OPENAI_API_KEY = None

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY: ").strip()

if not OPENAI_API_KEY:
    raise ValueError("An OpenAI API key is required to run this notebook.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [ ]:
# Set the chat model used throughout the notebook
llm_model = "gpt-4o-mini"

# Set the embedding model explicitly for reproducibility
embedding_model = "text-embedding-3-small"


In [ ]:
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_classic.chains import RetrievalQA
from langchain_classic.indexes import VectorstoreIndexCreator


In [ ]:
# download the same CSV the course uses
!wget -q https://raw.githubusercontent.com/Ryota-Kawamura/LangChain-for-LLM-Application-Development/main/OutdoorClothingCatalog_1000.csv


In [ ]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)

In [ ]:
embeddings = OpenAIEmbeddings(model=embedding_model)

index = VectorstoreIndexCreator(
    embedding=embeddings,
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])


In [ ]:
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

**Note**:

This notebook has been updated to use the current split LangChain packages:
- `langchain-openai`
- `langchain-community`
- `langchain-classic`

The chat model and embedding model are set explicitly in code, so the responses may differ slightly from older course videos or notebooks.


In [ ]:
llm = ChatOpenAI(temperature=0, model=llm_model)

response = index.query(query, llm=llm)


In [ ]:
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                   | Description Summary                                                                                     |
|----------------------------------------|--------------------------------------------------------------------------------------------------------|
| Sun Shield Shirt                       | High-performance sun shirt with UPF 50+ protection, moisture-wicking, abrasion-resistant, and soft fit. |
| Men's Tropical Plaid Short-Sleeve Shirt| Lightest hot-weather shirt with UPF 50+ protection, traditional fit, wrinkle-resistant, and cape venting. |
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable shirt with UPF 50+ protection, designed for fishing, moisture-wicking, and machine washable. |
| Tropical Breeze Shirt                  | Lightweight long-sleeve shirt with UPF 50+ protection, moisture-wicking, wrinkle-resistant, and traditional fit. |

## Step By Step

In [ ]:
loader = CSVLoader(file_path=file)

In [ ]:
docs = loader.load()

In [ ]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [ ]:
embed = embeddings.embed_query("Hi my name is Harrison")

In [ ]:
print(len(embed))

1536


In [ ]:
print(embed[:5])

[0.005710601806640625, -0.00586700439453125, -0.06268310546875, 0.0226898193359375, -0.051300048828125]


In [ ]:
db = DocArrayInMemorySearch.from_documents(
    docs,
    embeddings
)

In [ ]:
query = "Please suggest a shirt with sunblocking"

In [ ]:
docs = db.similarity_search(query)

In [ ]:
len(docs)

4

In [ ]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}, page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \n\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\n\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\n\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\n\nSun Protection That Won\'t Wear Off\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.')

In [ ]:
retriever = db.as_retriever()

In [ ]:
llm = ChatOpenAI(temperature = 0.0, model=llm_model)

In [ ]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))])


In [ ]:
response = llm.invoke(
    f"""{qdocs}
Question: Please list all your shirts with sun protection in a table in markdown and summarize each one."""
).content


In [ ]:
display(Markdown(response))

Here’s a table summarizing the shirts with sun protection:

| Name                                   | Description Summary                                                                                                                                                                                                                     |
|----------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Sun Shield Shirt**                  | High-performance sun shirt with UPF 50+ protection, blocking 98% of harmful UV rays. Slightly fitted, made of 78% nylon and 22% Lycra Xtra Life fiber. Moisture-wicking, abrasion-resistant, and comfortable over swimsuits.                  |
| **Men's Tropical Plaid Short-Sleeve Shirt** | Lightest hot-weather shirt rated UPF 50+ for sun protection. Traditional fit, made of 100% wrinkle-resistant polyester. Features front and back cape venting and two front bellows pockets for comfort and utility.                          |
| **Men's Plaid Tropic Shirt, Short-Sleeve** | Ultracomfortable shirt with UPF 50+ protection, originally designed for fishing. Made of 52% polyester and 48% nylon, it is wrinkle-free and moisture-evaporating. Includes front and back cape venting and two front bellows pockets.      |
| **Tropical Breeze Shirt**             | Lightweight, breathable long-sleeve shirt with UPF 50+ protection. Traditional fit, made of 71% nylon and 29% polyester. Features moisture-wicking, wrinkle-resistant fabric, front and back cape venting, and two front bellows pockets.      |

This table provides a concise overview of each shirt's features and benefits related to sun protection.

In [ ]:
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True
)

In [ ]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [ ]:
response = qa_stuff.run(query)



> Entering new RetrievalQA chain...

> Finished chain.


In [ ]:
display(Markdown(response))

Here is a table listing all the shirts with sun protection along with a summary of each:

| Name                                   | Description Summary                                                                                      |
|----------------------------------------|---------------------------------------------------------------------------------------------------------|
| Sun Shield Shirt                       | High-performance sun shirt with UPF 50+ protection, moisture-wicking, abrasion-resistant, and soft fit. |
| Men's Tropical Plaid Short-Sleeve Shirt| Light, traditional fit shirt made of 100% polyester with UPF 50+ protection and front/back cape venting. |
| Men's Plaid Tropic Shirt, Short-Sleeve | Comfortable, lightweight shirt with UPF 50+ protection, designed for fishing, and moisture-evaporating fabric. |
| Tropical Breeze Shirt                  | Lightweight, breathable long-sleeve shirt with UPF 50+ protection, moisture-wicking, and wrinkle-resistant fabric. |

In [ ]:
response = index.query(query, llm=llm)

In [ ]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

Reminder: Download your notebook to you local computer to save your work.